In [1]:
!pip install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 16.6 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "LiquidAI/LFM2-700M"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",        # best quality
    bnb_4bit_use_double_quant=True,   # improves precision
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.48G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [3]:
import gc
# Clear GPU memory
torch.cuda.empty_cache()
gc.collect()

124

In [5]:
import pandas as pd

start = 0
end = 3

# ========== قراءة ملف الداتا ==========
df = pd.read_excel("/content/drive/MyDrive/Datasets/base_data.xlsx")

df_sample = df.loc[start:end-1].copy()

# ========== تجهيز أعمدة جديدة ==========
df_sample["zero"] = ""
df_sample["static"] = ""
df_sample["dynamic"] = ""

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import os
from tqdm import tqdm


base_prompt = """
{POST}
"""

static_prompt = """
انت طبيب متخصص و دقيق في التشخيص.

مهمتك هي قراءة المنشور بعناية و تحديد المشكلة الصحية، كن واضح و مختصر.

أمثلة:
س: "لو عضني فأر و خرج دم ليس بالكثير ماذا علي ان افعل؟"
ج: "روح على مستشفى خذ ابرة تيتانوس"

س: "والدي ظهر عنده قرح فراش و بيهرش في جسمه و يعوره ، ازاي نعالجه؟"
ج: "شوفوا تحليل وظايف الكلى وظبطوا السكر أهم حاجة
ويدهن مكانها fucidin .. بس مهم يتقلب كل شوية على السرير ولو فيه امكانية لمرتبة هوائية يبقي أفضل
وتظبيط السكر."

س: "في كلكيعة صغيرة تحت الجلد ورا الأذن الشمال على العضمة كده ظهرت فجأه ولما اضغط عليها احس بألم ايه السبب؟"
ج: "في التهاب في الاذن نروح نكشف ممكن يكون الغدة الليمفاوية ملتهبة"

المنشور:
{POST}
"""

dynamic_prompt = """
انت طبيب متخصص في مجال {TYPE} و دقيق في التشخيص.

مهمتك هي قراءة المنشور بعناية و تحديد المشكلة الصحية، كن واضح و مختصر.

مع الأخذ في الاعتبار ان درجة خطورة المرض الآتي {SEVERITY}.

مصطلحات هامة: {DIAGNOSIS}.

أمثلة:
س: "لو عضني فأر و خرج دم ليس بالكثير ماذا علي ان افعل؟"
ج: "روح على مستشفى خذ ابرة تيتانوس"

س: "والدي ظهر عنده قرح فراش و بيهرش في جسمه و يعوره ، ازاي نعالجه؟"
ج: "شوفوا تحليل وظايف الكلى وظبطوا السكر أهم حاجة
ويدهن مكانها fucidin .. بس مهم يتقلب كل شوية على السرير ولو فيه امكانية لمرتبة هوائية يبقي أفضل
وتظبيط السكر."

س: "في كلكيعة صغيرة تحت الجلد ورا الأذن الشمال على العضمة كده ظهرت فجأه ولما اضغط عليها احس بألم ايه السبب؟"
ج: "في التهاب في الاذن نروح نكشف ممكن يكون الغدة الليمفاوية ملتهبة"

المنشور:
{POST}
"""


# ========== دالة للتوليد ==========
def generate_response(prompt_text):
    """
    توليد الاستجابة من النموذج المحلي
    """
    messages = [
        {"role": "user", "content": prompt_text}
    ]

    inputs = tokenizer.apply_chat_template(
      messages,
      add_generation_prompt=True,
      tokenize=True,
      return_dict=True,
      return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=100,
        temperature=0.6,
        top_p=0.95,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])

    return response.strip()

# ========== البدء ==========
# تأكد من تعريف df_sample, start, end قبل تشغيل هذا الكود
# مثال:
# df_sample = pd.read_excel("your_file.xlsx")
# start = 0
# end = 10

for i in tqdm(range(start, end)):
    row = df_sample.loc[i]

    post_text = "" if pd.isna(row["Post"]) else row["Post"]
    type_text = "" if pd.isna(row["Type"]) else row["Type"]
    severity_text = "" if pd.isna(row["Severity"]) else row["Severity"]
    diagnosis_text = "" if pd.isna(row["Diagnosis"]) else row["Diagnosis"]

    # ========== تحضير البرومبتات ==========
    prompt_post1 = (
        base_prompt
            .replace("{TYPE}", str(type_text))
            .replace("{SEVERITY}", str(severity_text))
            .replace("{DIAGNOSIS}", str(diagnosis_text))
            .replace("{POST}", str(post_text))
    )

    prompt_post2 = (
        static_prompt
            .replace("{TYPE}", str(type_text))
            .replace("{SEVERITY}", str(severity_text))
            .replace("{DIAGNOSIS}", str(diagnosis_text))
            .replace("{POST}", str(post_text))
    )

    prompt_post3 = (
        dynamic_prompt
            .replace("{TYPE}", str(type_text))
            .replace("{SEVERITY}", str(severity_text))
            .replace("{DIAGNOSIS}", str(diagnosis_text))
            .replace("{POST}", str(post_text))
    )

    # ========== توليد الاستجابات ==========
    print(f"⏳ Processing row {i}...")

    # Zero-shot
    response1 = generate_response(prompt_post1)
    df_sample.at[i, "zero"] = response1

    # Static
    response2 = generate_response(prompt_post2)
    df_sample.at[i, "static"] = response2

    # Dynamic
    response3 = generate_response(prompt_post3)
    df_sample.at[i, "dynamic"] = response3

    # ========== حفظ النتائج ==========
    df_sample2 = df_sample.head(i + 1)

    output_path = f"/content/drive/MyDrive/Datasets/Results/R_{start}_{i}.xlsx"
    df_sample2.to_excel(output_path, index=False)

    # حذف الملف السابق
    if i != start:
        prev_path = f"/content/drive/MyDrive/Datasets/Results/R_{start}_{i-1}.xlsx"
        if os.path.exists(prev_path):
            os.remove(prev_path)

    print(f"✓ Done row {i}")

print("Done!")

  0%|          | 0/3 [00:00<?, ?it/s]

⏳ Processing row 0...


 33%|███▎      | 1/3 [00:13<00:27, 13.92s/it]

✓ Done row 0
⏳ Processing row 1...


 67%|██████▋   | 2/3 [00:24<00:12, 12.19s/it]

✓ Done row 1
⏳ Processing row 2...


100%|██████████| 3/3 [00:36<00:00, 12.07s/it]

✓ Done row 2
Done!
